# NumPy and Pandas — Data Manipulation in Python
## From Arrays to DataFrames

---

## Table of Contents

**NumPy**
1. Arrays — Creation and Properties
2. Indexing, Slicing, and Boolean Masks
3. Vectorized Operations and Broadcasting
4. Mathematical Functions and Linear Algebra
5. Random Module and Statistics

**Pandas**
6. Series and DataFrame Fundamentals
7. Indexing — loc, iloc, Boolean Filtering
8. Handling Missing Data
9. GroupBy and Aggregation
10. Merge, Join, and Reshape
11. String Operations, Categoricals, and Time Series
12. Performance — Vectorization vs apply

---


In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 100)

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

# Section 1 — Arrays: Creation and Properties

## Concept

NumPy's `ndarray` is a fixed-type, contiguous-memory array.
It is the foundation of nearly all scientific Python: pandas, scikit-learn, PyTorch, and OpenCV all use it internally.

## Technical Deep Dive

| Attribute | Meaning |
|---|---|
| `ndim` | Number of dimensions |
| `shape` | Tuple of sizes per dimension |
| `size` | Total number of elements |
| `dtype` | Element data type |
| `itemsize` | Bytes per element |
| `nbytes` | Total memory in bytes |

**Key dtypes:** `int32`, `int64`, `float32`, `float64`, `bool`, `complex128`, `object`.
Prefer `float32` for ML (half the memory of `float64`).


In [ ]:
# Creation from Python structures
a1 = np.array([1, 2, 3, 4, 5])
a2 = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)

print('1D:', a1, a1.shape, a1.dtype)
print('2D:', a2.shape, a2.dtype, f'{a2.nbytes} bytes')

# Factory functions
print('\nzeros 3x4:');     print(np.zeros((3, 4)))
print('\nones 2x3:');      print(np.ones((2, 3)))
print('\neye 3x3:');       print(np.eye(3))
print('\nfull 2x2=7:');    print(np.full((2, 2), 7))

In [ ]:
# Range-like creators
print('arange(0,10,2):  ', np.arange(0, 10, 2))
print('linspace(0,1,5): ', np.linspace(0, 1, 5))
print('logspace(0,3,4): ', np.logspace(0, 3, 4))

# Reshape
r = np.arange(12).reshape(3, 4)
print('\nreshaped 3x4:\n', r)

# Memory efficiency comparison
list_1m = list(range(1_000_000))
arr_1m  = np.arange(1_000_000, dtype=np.int32)
import sys
print(f'\nPython list (1M ints): {sys.getsizeof(list_1m) / 1e6:.1f} MB')
print(f'NumPy array (1M int32): {arr_1m.nbytes / 1e6:.1f} MB')

## Exercises

1. Create a 5x5 identity matrix as `float32`.
2. Create an array of 20 evenly spaced values between -π and π.
3. Create a 4x4 matrix filled with the value 42.
4. Create an array `[0, 0.1, 0.2, ..., 1.0]` using `linspace`.

## Mini Challenge

Create a 10x10 "checkerboard" array: 0s and 1s in a checkerboard pattern (no loops).

## Best Practices

- Specify `dtype` explicitly when memory matters.
- Use `np.empty` (uninitialized) when you will fill every element — fastest allocation.
- Avoid `dtype=object` — you lose all vectorization benefits.

## Common Mistakes

- `np.array([1, 2, 3.0])` silently upcasts everything to `float64`.
- `np.arange` with float steps has floating-point precision issues; use `linspace` instead.
- Confusing `shape=(n,)` (1D) with `shape=(n,1)` (column vector).

## Summary

- `ndarray` is fixed-type, contiguous memory — far more efficient than Python lists.
- Key factories: `zeros`, `ones`, `eye`, `full`, `arange`, `linspace`.
- Always check `.shape`, `.dtype`, `.nbytes` when debugging.

---


### Exercise and Challenge Solutions — Section 1


In [ ]:
print(np.eye(5, dtype=np.float32))
print(np.linspace(-np.pi, np.pi, 20))
print(np.full((4, 4), 42))
print(np.linspace(0, 1, 11))

In [ ]:
# Checkerboard: tile [[0,1],[1,0]] pattern
checkerboard = np.zeros((10, 10), dtype=int)
checkerboard[1::2, ::2]  = 1
checkerboard[::2,  1::2] = 1
print(checkerboard)

# Section 2 — Indexing, Slicing, and Boolean Masks

## Concept

NumPy supports three indexing modes: basic slicing (returns a **view**),
fancy indexing (returns a **copy**), and boolean masking.

## Technical Deep Dive

```python
a[start:stop:step]          # basic slice — view
a[[0, 2, 4]]                # fancy index — copy
a[a > 5]                    # boolean mask — copy
a[row_idx, col_idx]         # multi-dimensional
a[..., 0]                   # ellipsis — all dims before last
```

**View vs copy is critical:** modifying a view modifies the original.
Use `.copy()` to avoid accidental mutation.


In [ ]:
arr = np.arange(20).reshape(4, 5)
print('Original:\n', arr)
print('Row 1:         ', arr[1])
print('Col 2:         ', arr[:, 2])
print('Rows 1-2, cols 1-3:\n', arr[1:3, 1:4])
print('Every 2nd row: \n', arr[::2])
print('Last 2 cols:   \n', arr[:, -2:])

In [ ]:
# Fancy indexing
print('Rows [0, 3]:\n', arr[[0, 3]])
print('Specific elements (0,0),(1,2),(3,4):', arr[[0,1,3], [0,2,4]])

# Boolean mask
mask = arr > 10
print('\nMask (>10):\n', mask)
print('Values > 10:', arr[mask])

# Combined condition
print('8 < x < 15:', arr[(arr > 8) & (arr < 15)])

In [ ]:
# View vs copy
original = np.array([1, 2, 3, 4, 5])
view = original[1:4]      # slice = view
copy = original[[1,2,3]]  # fancy = copy

view[0] = 99
copy[0] = 77

print('original after view mutation:', original)  # changed!
print('original after copy mutation:', original)  # unchanged

## Exercises

1. From a 6x6 matrix (`np.arange(36).reshape(6,6)`), extract the center 4x4 block.
2. Select all even numbers from `np.arange(1, 21)`.
3. Replace all negative values in `np.random.randn(5, 5)` with 0 using a boolean mask.
4. Reverse a 1D array without using `np.flip`.

## Mini Challenge

Given `arr = np.random.randint(0, 100, size=(8, 8))`, find the row and column indices
of all values greater than 80. Return as a list of `(row, col)` tuples.

## Summary

- Slices return views; fancy indexing and boolean masks return copies.
- Boolean masks are the idiomatic way to filter arrays.
- `np.where(cond, x, y)` is a vectorized if-else.

---


### Exercise and Challenge Solutions — Section 2


In [ ]:
m = np.arange(36).reshape(6, 6)
print('Center 4x4:\n', m[1:5, 1:5])
r = np.arange(1, 21)
print('Evens:', r[r % 2 == 0])
rnd = np.random.randn(5, 5)
rnd[rnd < 0] = 0
print('Negatives replaced:\n', rnd.round(2))
a = np.arange(10)
print('Reversed:', a[::-1])

In [ ]:
arr = np.random.randint(0, 100, size=(8, 8))
rows, cols = np.where(arr > 80)
positions = list(zip(rows.tolist(), cols.tolist()))
print('Values > 80 at:', positions)
print('Values:', arr[rows, cols])

# Section 3 — Vectorized Operations and Broadcasting

## Concept

**Vectorized operations** apply element-wise without Python loops — implemented in C, orders of magnitude faster.
**Broadcasting** allows arrays of different shapes to be combined by implicitly expanding dimensions.

## Technical Deep Dive

**Broadcasting rules (applied right-to-left on shapes):**
1. If arrays have different number of dims, prepend 1s to the smaller shape.
2. Dimensions of size 1 are stretched to match the other.
3. If sizes differ and neither is 1, error.

```
Shape (3, 1) + Shape (1, 4) → Shape (3, 4)   ✓
Shape (3,)   + Shape (3, 3) → Shape (3, 3)   ✓  (3,) becomes (1,3))
Shape (2,)   + Shape (3,)   →  ERROR          ✗
```


In [ ]:
import time

n = 1_000_000
a = np.random.rand(n)
b = np.random.rand(n)

# Loop version
t0 = time.perf_counter()
result_loop = [a[i] * b[i] for i in range(n)]
t_loop = time.perf_counter() - t0

# Vectorized version
t0 = time.perf_counter()
result_vec = a * b
t_vec = time.perf_counter() - t0

print(f'Loop:       {t_loop*1000:.1f} ms')
print(f'Vectorized: {t_vec*1000:.1f} ms')
print(f'Speedup:    {t_loop/t_vec:.0f}x')

In [ ]:
# Broadcasting examples
# Subtract row mean from each row (normalize)
matrix = np.array([[1,2,3],[4,5,6],[7,8,9]], dtype=float)
row_means = matrix.mean(axis=1, keepdims=True)  # shape (3,1)
normalized = matrix - row_means                  # broadcasts (3,1) across (3,3)
print('Row-normalized:\n', normalized)

# Outer product via broadcasting
x = np.array([1, 2, 3])          # shape (3,)
y = np.array([10, 20, 30, 40])   # shape (4,)
outer = x[:, np.newaxis] * y     # (3,1) * (4,) = (3,4)
print('\nOuter product (3x4):\n', outer)

In [ ]:
# Universal functions (ufuncs)
x = np.linspace(0, 2*np.pi, 6)
print('x:    ', x.round(3))
print('sin:  ', np.sin(x).round(3))
print('cos:  ', np.cos(x).round(3))
print('exp:  ', np.exp([0, 1, 2]).round(4))
print('log:  ', np.log([1, np.e, np.e**2]).round(4))

# np.where — vectorized conditional
arr = np.array([-3, -1, 0, 2, 5])
print('\nnp.where (clip negatives to 0):', np.where(arr < 0, 0, arr))

# np.clip
print('np.clip [0, 3]:               ', np.clip(arr, 0, 3))

## Exercises

1. Normalize a matrix to have zero mean and unit variance (z-score) per column.
2. Compute the pairwise Euclidean distance between 5 points in 3D (each point is a row in a 5x3 array).
3. Use `np.where` to implement ReLU: `max(0, x)` on a random array.
4. Compute the sigmoid function `1 / (1 + exp(-x))` vectorized.

## Mini Challenge

Without loops, compute the outer product of two vectors AND the inner product, then verify with `np.dot` and `np.outer`.
Then time a matrix multiply of two 1000x1000 matrices using `@` vs a Python loop.

## Summary

- Vectorized ops run in C — 100-1000x faster than Python loops.
- Broadcasting eliminates the need for explicit replication.
- Use `keepdims=True` when reducing to preserve shape for broadcasting.
- `np.where`, `np.clip`, `np.abs` are your vectorized conditionals.

---


### Exercise and Challenge Solutions — Section 3


In [ ]:
# 1. Z-score normalization per column
data = np.random.randn(5, 4) * 10 + 50
z = (data - data.mean(axis=0)) / data.std(axis=0)
print('Z-score (col means ~0):', z.mean(axis=0).round(10))

# 2. Pairwise distances
pts = np.random.rand(5, 3)
diff = pts[:, np.newaxis, :] - pts[np.newaxis, :, :]  # (5,5,3)
dists = np.sqrt((diff**2).sum(axis=-1))
print('\nPairwise distances:\n', dists.round(3))

# 3. ReLU
x = np.random.randn(6)
relu = np.where(x < 0, 0, x)
print('\nReLU:', relu.round(3))

# 4. Sigmoid
sigmoid = lambda x: 1 / (1 + np.exp(-x))
print('Sigmoid:', sigmoid(np.array([-2, -1, 0, 1, 2])).round(4))

In [ ]:
# Mini Challenge: outer/inner products + matrix multiply timing
# YOUR CODE HERE

# Section 4 — Mathematical Functions and Linear Algebra

## Concept

NumPy provides a complete suite of linear algebra operations via `np.linalg`.
These are used in ML (solving linear systems, PCA, SVD).

## Technical Deep Dive

| Operation | NumPy |
|---|---|
| Matrix multiply | `A @ B` or `np.matmul(A, B)` |
| Dot product | `np.dot(a, b)` |
| Transpose | `A.T` |
| Determinant | `np.linalg.det(A)` |
| Inverse | `np.linalg.inv(A)` |
| Eigenvalues | `np.linalg.eig(A)` |
| SVD | `np.linalg.svd(A)` |
| Solve Ax=b | `np.linalg.solve(A, b)` |
| Least squares | `np.linalg.lstsq(A, b)` |
| Norms | `np.linalg.norm(v, ord=2)` |


In [ ]:
A = np.array([[2., 1.], [5., 3.]])
b = np.array([4., 7.])

print('A:\n', A)
print('det(A):', np.linalg.det(A))
print('inv(A):\n', np.linalg.inv(A))

# Solve Ax = b
x = np.linalg.solve(A, b)
print('\nSolution x (Ax=b):', x)
print('Verify Ax:', A @ x)  # should equal b

In [ ]:
# SVD — foundation of PCA
M = np.random.randn(4, 3)
U, S, Vt = np.linalg.svd(M, full_matrices=False)
print('M shape:', M.shape)
print('U:', U.shape, 'S:', S.shape, 'Vt:', Vt.shape)
print('Singular values:', S.round(4))

# Reconstruct M from SVD
M_reconstructed = U @ np.diag(S) @ Vt
print('Reconstruction error:', np.max(np.abs(M - M_reconstructed)))

# Low-rank approximation (keep only top k singular values)
k = 2
M_approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
print(f'Rank-{k} approximation error:', np.linalg.norm(M - M_approx).round(4))

In [ ]:
# Reduction functions
arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
print('sum all:  ', arr.sum())
print('sum cols: ', arr.sum(axis=0))   # collapse rows
print('sum rows: ', arr.sum(axis=1))   # collapse cols
print('cumsum:   ', arr.ravel().cumsum())
print('mean:     ', arr.mean())
print('std:      ', arr.std().round(4))
print('argmax:   ', arr.argmax(), '→ row', *np.unravel_index(arr.argmax(), arr.shape))

## Exercises

1. Compute the L1 and L2 norms of vector `[3, 4, 0]`.
2. Generate a random 3x3 matrix, compute its eigenvalues and eigenvectors.
3. Solve the system: 3x + 2y = 12, x - y = 1.
4. Compute the column-wise cumulative sum of a 4x4 matrix.

## Mini Challenge

Implement manual PCA (no sklearn): center the data, compute the covariance matrix,
then use `np.linalg.eigh` to get eigenvectors, project data onto top 2 components.
Test on a 100x4 random dataset.

## Summary

- `np.linalg` provides production-grade linear algebra backed by LAPACK/BLAS.
- SVD is the most numerically stable factorization.
- Use `np.linalg.solve` instead of `inv(A) @ b` — more numerically stable.
- `axis` parameter controls which dimension to collapse in reductions.

---


### Exercise and Challenge Solutions — Section 4


In [ ]:
v = np.array([3., 4., 0.])
print('L1:', np.linalg.norm(v, ord=1))
print('L2:', np.linalg.norm(v, ord=2))

M = np.random.randn(3, 3)
vals, vecs = np.linalg.eig(M)
print('\nEigenvalues:', vals.round(3))

A = np.array([[3., 2.], [1., -1.]])
b = np.array([12., 1.])
print('\nx, y =', np.linalg.solve(A, b).round(4))

m = np.arange(1, 17, dtype=float).reshape(4, 4)
print('\nCumulative col sum:\n', m.cumsum(axis=0))

In [ ]:
# Manual PCA
np.random.seed(0)
X = np.random.randn(100, 4)

# 1. Center
X_centered = X - X.mean(axis=0)

# 2. Covariance matrix
cov = np.cov(X_centered.T)  # (4, 4)

# 3. Eigenvectors (eigh for symmetric matrices)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
# Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvectors = eigenvectors[:, idx]
eigenvalues  = eigenvalues[idx]

# 4. Project onto top 2
X_pca = X_centered @ eigenvectors[:, :2]

print('Original shape:', X.shape)
print('PCA shape:     ', X_pca.shape)
print('Explained variance ratio:', (eigenvalues[:2] / eigenvalues.sum()).round(3))

# Section 5 — Random Module and Statistics

## Concept

`np.random` (and the newer `np.random.default_rng`) provides random number generation
and statistical sampling. Used heavily in simulation, ML weight init, and bootstrapping.

## Technical Deep Dive

| Function | Description |
|---|---|
| `rng.random(size)` | Uniform [0, 1) |
| `rng.integers(low, high, size)` | Random integers |
| `rng.normal(mu, sigma, size)` | Gaussian distribution |
| `rng.uniform(low, high, size)` | Uniform [low, high) |
| `rng.choice(a, size, replace)` | Sample from array |
| `rng.shuffle(arr)` | Shuffle in-place |
| `rng.permutation(arr)` | Return shuffled copy |
| `rng.binomial(n, p, size)` | Binomial distribution |
| `rng.exponential(scale, size)` | Exponential distribution |

**Best practice:** use `np.random.default_rng(seed)` — the new Generator API is reproducible and faster.


In [ ]:
rng = np.random.default_rng(42)

print('Uniform [0,1):', rng.random(5).round(4))
print('Integers [0,10):', rng.integers(0, 10, size=8))
print('Normal(0,1):', rng.normal(0, 1, size=5).round(4))
print('Choice from [a,b,c,d]:', rng.choice(['a','b','c','d'], size=6, replace=True))

# Reproducibility
rng1 = np.random.default_rng(99)
rng2 = np.random.default_rng(99)
print('\nSame seed → same output:', np.allclose(rng1.random(5), rng2.random(5)))

In [ ]:
# Descriptive statistics
data = rng.normal(loc=50, scale=15, size=1000)

print(f'Mean:     {data.mean():.2f}')
print(f'Median:   {np.median(data):.2f}')
print(f'Std:      {data.std():.2f}')
print(f'Variance: {data.var():.2f}')
print(f'P25:      {np.percentile(data, 25):.2f}')
print(f'P75:      {np.percentile(data, 75):.2f}')
print(f'IQR:      {np.percentile(data, 75) - np.percentile(data, 25):.2f}')
print(f'Min:      {data.min():.2f}')
print(f'Max:      {data.max():.2f}')

In [ ]:
# Bootstrap confidence interval for the mean
sample = rng.normal(100, 20, size=50)
n_bootstrap = 10_000

boot_means = np.array([
    rng.choice(sample, size=len(sample), replace=True).mean()
    for _ in range(n_bootstrap)
])

ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f'Sample mean: {sample.mean():.2f}')
print(f'95% bootstrap CI: [{ci_low:.2f}, {ci_high:.2f}]')

## Summary — NumPy

- Use `np.random.default_rng(seed)` for reproducible, modern random generation.
- Core distributions: normal, uniform, integers, binomial, exponential.
- Descriptive stats: `mean`, `median`, `std`, `percentile`, `var`.
- Bootstrapping = resampling with replacement for robust confidence intervals.

---

# PANDAS

---


# Section 6 — Series and DataFrame Fundamentals

## Concept

- **Series** = 1D labeled array (like a column with an index).
- **DataFrame** = 2D table of Series sharing an index.

Both support heterogeneous dtypes, missing values, and rich label-based operations.

## Technical Deep Dive

```python
pd.Series(data, index=...)       # from list, dict, ndarray, scalar
pd.DataFrame(data, columns=...)  # from dict, list of dicts, ndarray, CSV
```

**Key DataFrame attributes:**
`shape`, `dtypes`, `columns`, `index`, `values` (returns ndarray), `info()`, `describe()`


In [ ]:
# Series
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'], name='values')
print('Series:'); print(s)
print('s["b"]:', s['b'])
print('s * 2:\n', s * 2)

# DataFrame from dict
df = pd.DataFrame({
    'name':       ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace'],
    'department': ['Eng',   'Eng',  'Mktg',  'Sales', 'Eng',  'HR',   'Fin'],
    'salary':     [95000,   85000,  75000,   65000,   90000,  60000,  80000],
    'years':      [5,       4,      6,       3,       5,      2,      4],
    'active':     [True,    True,   True,    False,   True,   True,   True]
})
print('\nDataFrame:'); print(df)

In [ ]:
print(df.info())
print()
print(df.describe())
print()
print('dtypes:\n', df.dtypes)
print('shape:', df.shape)
print('columns:', df.columns.tolist())

In [ ]:
# Column operations
df['monthly_salary'] = df['salary'] / 12
df['seniority'] = pd.cut(df['years'], bins=[0,2,4,10], labels=['Junior','Mid','Senior'])

print(df[['name', 'salary', 'monthly_salary', 'seniority']])

## Exercises

1. From `df`, select only the `name` and `salary` columns.
2. Add a column `salary_above_avg` (boolean) — True if salary > mean salary.
3. Rename column `years` to `tenure`.
4. Drop the `monthly_salary` column.

## Mini Challenge

Create a DataFrame from a list of 20 dictionaries representing students with random `score` (50-100) and `grade` (A/B/C/D/F based on score). Compute a summary table showing count and mean score per grade.

## Summary

- Series = labeled 1D array; DataFrame = labeled 2D table.
- Create from dicts, lists, ndarrays, CSV, SQL.
- `info()` and `describe()` are your first stops when exploring data.
- Column assignment creates new columns; use `pd.cut` / `pd.qcut` for binning.

---


### Exercise and Challenge Solutions — Section 6


In [ ]:
print(df[['name', 'salary']])
df['salary_above_avg'] = df['salary'] > df['salary'].mean()
print(df[['name', 'salary', 'salary_above_avg']])
df = df.rename(columns={'years': 'tenure'})
df = df.drop(columns=['monthly_salary'])
print(df.columns.tolist())

In [ ]:
rng2 = np.random.default_rng(7)
scores = rng2.integers(50, 101, size=20)

def score_to_grade(s):
    if s >= 90: return 'A'
    if s >= 80: return 'B'
    if s >= 70: return 'C'
    if s >= 60: return 'D'
    return 'F'

students = pd.DataFrame({'score': scores})
students['grade'] = students['score'].map(score_to_grade)
summary = students.groupby('grade').agg(count=('score','count'), mean_score=('score','mean'))
print(summary)

# Section 7 — Indexing: loc, iloc, Boolean Filtering

## Concept

Pandas has two primary indexers:
- `loc` — **label**-based (row/column names)
- `iloc` — **position**-based (integer offsets)

## Technical Deep Dive

| Accessor | Selects by | Example |
|---|---|---|
| `df['col']` | Column by name | `df['salary']` |
| `df[['a','b']]` | Multiple columns | `df[['name','dept']]` |
| `df.loc[row, col]` | Label | `df.loc[2, 'name']` |
| `df.iloc[row, col]` | Position | `df.iloc[0, 1]` |
| `df.at[row, col]` | Single label value (fast) | `df.at[3, 'salary']` |
| `df.iat[row, col]` | Single position value (fast) | `df.iat[0, 0]` |
| `df[mask]` | Boolean filter | `df[df.salary > 80000]` |

**`loc` slices are inclusive on both ends; `iloc` slices exclude the right end.**


In [ ]:
# loc — label based
print('Row at index label 2:'); print(df.loc[2])
print('\nRows 1-3, cols name+salary:')
print(df.loc[1:3, ['name', 'salary']])

# iloc — position based
print('\nFirst 3 rows, last 2 cols:')
print(df.iloc[:3, -2:])

In [ ]:
# Boolean filtering
high_earners = df[df['salary'] > 80000]
print('High earners:'); print(high_earners[['name','department','salary']])

# Multiple conditions
eng_seniors = df[(df['department'] == 'Eng') & (df['tenure'] >= 4)]
print('\nEng seniors:'); print(eng_seniors[['name','tenure','salary']])

# isin
selected_depts = df[df['department'].isin(['Eng', 'Fin'])]
print('\nEng or Fin:'); print(selected_depts[['name','department']])

# query() — readable alternative
print('\nquery() style:')
print(df.query('salary > 75000 and active == True')[['name','salary']])

## Exercises

1. Select the 3rd to 5th rows (by position) using `iloc`.
2. Filter employees who are active AND have salary below 75000.
3. Use `.query()` to find employees in 'Eng' with tenure > 3.
4. Update the salary of employee at index 3 to 70000 using `.at`.

## Summary

- `loc` = labels, `iloc` = positions — never mix them.
- Boolean masks compose with `&`, `|`, `~` (not `and`, `or`, `not`).
- `.query()` is readable for complex filters.
- Use `.at` / `.iat` for single-cell access — fastest.

---


### Exercise and Challenge Solutions — Section 7


In [ ]:
print(df.iloc[2:5])
print(df[(df['active'] == True) & (df['salary'] < 75000)][['name','salary','active']])
print(df.query('department == "Eng" and tenure > 3')[['name','tenure']])
df.at[3, 'salary'] = 70000
print('Updated:', df.at[3, 'salary'])

# Section 8 — Handling Missing Data

## Concept

Real-world data has missing values. Pandas represents them as `NaN` (float) or `pd.NA`.
Detecting, dropping, and filling missing data correctly is essential for analysis.

## Technical Deep Dive

| Method | Description |
|---|---|
| `df.isna()` | Boolean mask of missing values |
| `df.notna()` | Opposite of isna |
| `df.isna().sum()` | Count missing per column |
| `df.dropna()` | Drop rows/cols with NaN |
| `df.fillna(value)` | Fill NaN with scalar or dict |
| `df.fillna(method='ffill')` | Forward fill |
| `df.fillna(method='bfill')` | Backward fill |
| `df.interpolate()` | Linear (or other) interpolation |

**Imputation strategies:** mean/median (numerical), mode (categorical), model-based.


In [ ]:
# Create dataset with missing values
df_miss = pd.DataFrame({
    'A': [1., 2., np.nan, 4., 5.],
    'B': [np.nan, 2., 3., np.nan, 5.],
    'C': ['x', np.nan, 'z', 'w', np.nan],
    'D': [10., 20., 30., 40., 50.]
})
print('Dataset with NaN:'); print(df_miss)
print('\nMissing count:'); print(df_miss.isna().sum())
print('\nMissing %:'); print((df_miss.isna().mean() * 100).round(1))

In [ ]:
# Drop rows where any value is NaN
print('After dropna():'); print(df_miss.dropna())

# Drop cols with more than 20% missing
thresh = int(len(df_miss) * 0.8)
print('\nAfter dropping sparse cols:'); print(df_miss.dropna(axis=1, thresh=thresh))

# Fill strategies
df_filled = df_miss.copy()
df_filled['A'] = df_filled['A'].fillna(df_filled['A'].median())
df_filled['B'] = df_filled['B'].fillna(method='ffill')
df_filled['C'] = df_filled['C'].fillna('unknown')
print('\nAfter filling:'); print(df_filled)

## Exercises

1. Create a 6x4 DataFrame with 30% of values randomly set to NaN. Report missing counts.
2. Fill numeric NaN columns with their mean, categorical with 'missing'.
3. Drop rows where more than 2 columns are NaN.
4. Use `interpolate()` to fill NaN in a time series-like numeric column.

## Summary

- `isna()`, `dropna()`, `fillna()` are the core missing data toolkit.
- Never use `== NaN` — always use `isna()`.
- Choose fill strategy based on domain knowledge, not just convenience.
- `interpolate()` is powerful for time series data.

---


### Exercise and Challenge Solutions — Section 8


In [ ]:
rng3 = np.random.default_rng(10)
data = rng3.random((6, 4))
mask = rng3.random((6, 4)) < 0.3
df_ex = pd.DataFrame(data, columns=['n1','n2','cat','n3'])
df_ex[mask] = np.nan
print('Missing counts:'); print(df_ex.isna().sum())

for col in ['n1','n2','n3']:
    df_ex[col] = df_ex[col].fillna(df_ex[col].mean())
df_ex['cat'] = df_ex['cat'].fillna('missing')
print('\nAfter fill:'); print(df_ex.round(3))

# Drop rows with >2 NaN before the fill
df_ex2 = pd.DataFrame(data, columns=['n1','n2','cat','n3'])
df_ex2[mask] = np.nan
print('\nDrop rows with >2 NaN:'); print(df_ex2.dropna(thresh=2))

# Interpolate
ts = pd.Series([1., np.nan, np.nan, 4., np.nan, 6.])
print('\nInterpolated:', ts.interpolate().tolist())

# Section 9 — GroupBy and Aggregation

## Concept

`groupby` is pandas' split-apply-combine engine:
1. **Split** data into groups by one or more keys.
2. **Apply** a function to each group.
3. **Combine** results into a new DataFrame.

## Technical Deep Dive

```python
df.groupby('col').agg({'num_col': ['mean', 'sum', 'std']})
df.groupby('col')['num'].transform('mean')   # broadcast back to original index
df.groupby('col').filter(lambda g: g['sal'].mean() > 75000)  # keep whole groups
df.groupby('col').apply(custom_func)          # arbitrary function
```

**`agg` vs `transform`:**
- `agg` reduces each group to one row.
- `transform` returns a result with the same shape as the input (for enriching the original df).


In [ ]:
# Reload clean employee df
df_emp = pd.DataFrame({
    'name':       ['Alice','Bob','Carol','David','Eve','Frank','Grace','Henry','Iris','Jack'],
    'department': ['Eng','Eng','Mktg','Sales','Eng','HR','Fin','Sales','Mktg','Eng'],
    'salary':     [95000,85000,75000,65000,90000,60000,80000,70000,72000,88000],
    'tenure':     [5,4,6,3,5,2,4,3,5,4]
})

# Basic groupby
print('=== Dept stats ===')
print(df_emp.groupby('department').agg(
    headcount=('name','count'),
    avg_salary=('salary','mean'),
    total_payroll=('salary','sum'),
    avg_tenure=('tenure','mean')
).round(0).sort_values('avg_salary', ascending=False))

In [ ]:
# transform — broadcast group stat back to original df
df_emp['dept_avg_salary'] = df_emp.groupby('department')['salary'].transform('mean')
df_emp['salary_vs_dept']  = df_emp['salary'] - df_emp['dept_avg_salary']
print(df_emp[['name','department','salary','dept_avg_salary','salary_vs_dept']].round(0))

In [ ]:
# filter — keep only groups meeting a criterion
large_depts = df_emp.groupby('department').filter(lambda g: len(g) >= 2)
print('Departments with >= 2 employees:')
print(large_depts[['name','department']].sort_values('department'))

# apply — custom function per group
def top_earner(group):
    return group.nlargest(1, 'salary')[['name','salary']]

print('\nTop earner per department:')
print(df_emp.groupby('department').apply(top_earner).reset_index(level=1, drop=True))

## Exercises

1. Group by department and compute min, max, and median salary.
2. Add a column `salary_rank` showing each employee's rank within their department (1 = highest).
3. Filter to keep only departments where total payroll exceeds 200000.
4. Group by `tenure` (rounded to 2-year bins using `pd.cut`) and compute average salary.

## Mini Challenge

Compute a normalized salary score per department: `(salary - dept_min) / (dept_max - dept_min)`,
so each department's salary range maps to [0, 1].

## Summary

- `groupby` implements split-apply-combine.
- `agg` reduces; `transform` broadcasts; `filter` selects groups; `apply` is fully flexible.
- Named aggregation `agg(new_col=('col', 'func'))` is the modern preferred syntax.

---


### Exercise and Challenge Solutions — Section 9


In [ ]:
print(df_emp.groupby('department')['salary'].agg(['min','max','median']))

df_emp['salary_rank'] = df_emp.groupby('department')['salary'].rank(method='min', ascending=False)
print(df_emp[['name','department','salary','salary_rank']].sort_values(['department','salary_rank']))

big = df_emp.groupby('department').filter(lambda g: g['salary'].sum() > 200000)
print(big['department'].unique())

df_emp['tenure_bin'] = pd.cut(df_emp['tenure'], bins=[0,2,4,10], labels=['0-2','3-4','5+'])
print(df_emp.groupby('tenure_bin', observed=False)['salary'].mean().round(0))

In [ ]:
def normalize(g):
    mn, mx = g['salary'].min(), g['salary'].max()
    return (g['salary'] - mn) / (mx - mn) if mx != mn else pd.Series(0.5, index=g.index)

df_emp['salary_norm'] = df_emp.groupby('department', group_keys=False).apply(normalize)
print(df_emp[['name','department','salary','salary_norm']].round(3))

# Section 10 — Merge, Join, and Reshape

## Concept

Combining DataFrames is a core data engineering task.
Reshaping converts between wide and long formats.

## Technical Deep Dive

| Operation | Use Case |
|---|---|
| `pd.merge(left, right, on, how)` | SQL-style JOIN |
| `df.join(other, on, how)` | Join on index |
| `pd.concat([df1, df2])` | Stack DataFrames vertically or horizontally |
| `df.pivot(index, columns, values)` | Long → wide |
| `df.melt(id_vars, value_vars)` | Wide → long |
| `df.pivot_table(...)` | Grouped pivot with aggregation |
| `df.stack()` / `df.unstack()` | Index-level reshaping |

`how`: `'inner'`, `'left'`, `'right'`, `'outer'`, `'cross'`


In [ ]:
employees = pd.DataFrame({
    'emp_id':  [1,2,3,4,5],
    'name':    ['Alice','Bob','Carol','David','Eve'],
    'dept_id': [10,10,20,30,10]
})
departments = pd.DataFrame({
    'dept_id':  [10,20,30,40],
    'dept_name':['Engineering','Marketing','Sales','HR'],
    'budget':   [500000,200000,300000,150000]
})

# INNER JOIN
merged = pd.merge(employees, departments, on='dept_id', how='inner')
print('Inner join:'); print(merged)

# LEFT JOIN — keep all employees even without dept
left = pd.merge(employees, departments, on='dept_id', how='left')
print('\nLeft join:'); print(left)

In [ ]:
# concat
q1 = pd.DataFrame({'month': ['Jan','Feb','Mar'], 'sales': [100,120,90]})
q2 = pd.DataFrame({'month': ['Apr','May','Jun'], 'sales': [110,130,115]})
full = pd.concat([q1, q2], ignore_index=True)
print('Concatenated:'); print(full)

# pivot_table
data = pd.DataFrame({
    'region':  ['East','East','West','West','East','West'],
    'product': ['A','B','A','B','A','B'],
    'sales':   [100,150,200,80,120,90]
})
pivot = data.pivot_table(values='sales', index='region', columns='product', aggfunc='sum')
print('\nPivot table:'); print(pivot)

In [ ]:
# melt — wide to long
wide = pd.DataFrame({
    'employee': ['Alice','Bob','Carol'],
    'Q1_sales': [100,120,90],
    'Q2_sales': [110,130,95],
    'Q3_sales': [105,140,88]
})
print('Wide format:'); print(wide)

long = wide.melt(id_vars='employee', var_name='quarter', value_name='sales')
print('\nLong (melted) format:'); print(long)

## Exercises

1. Merge `employees` and `departments` with a right join. What happens to dept 40 (HR)?
2. `pd.concat` three single-row DataFrames and reset the index.
3. Pivot the `data` DataFrame above to show mean sales per region per product.
4. Melt `wide` into long format, then compute mean sales per quarter.

## Summary

- `pd.merge` is the go-to for SQL-style joins.
- `pd.concat` stacks DataFrames; use `ignore_index=True` to reset labels.
- `pivot_table` = groupby + reshape in one step.
- `melt` converts wide → long for tidy data format.

---


### Exercise and Challenge Solutions — Section 10


In [ ]:
right = pd.merge(employees, departments, on='dept_id', how='right')
print('Right join (HR appears with NaN employees):'); print(right)

three = pd.concat([
    pd.DataFrame({'x': [1]}),
    pd.DataFrame({'x': [2]}),
    pd.DataFrame({'x': [3]})
], ignore_index=True)
print('\nConcatenated:'); print(three)

print('\nMean pivot:'); print(data.pivot_table(values='sales', index='region', columns='product', aggfunc='mean'))

long2 = wide.melt(id_vars='employee', var_name='quarter', value_name='sales')
print('\nMean per quarter:'); print(long2.groupby('quarter')['sales'].mean())

# Section 11 — String Operations, Categoricals, and Time Series

## Concept

- **String accessor** `str.` — vectorized string methods on object columns.
- **Categorical** — memory-efficient representation of low-cardinality columns.
- **Time series** — `DatetimeIndex` enables date-aware slicing and resampling.


In [ ]:
names = pd.Series(['alice johnson', 'BOB SMITH', 'carol  white', 'david-lee'])

print('upper:  ', names.str.upper().tolist())
print('title:  ', names.str.title().tolist())
print('strip:  ', names.str.strip().tolist())
print('replace:', names.str.replace('-', ' ').tolist())
print('contains "o":', names.str.contains('o', case=False).tolist())
print('split:  '); print(names.str.split(' ', expand=True))
print('len:    ', names.str.len().tolist())

In [ ]:
# Categorical — memory savings for repeated strings
n = 100_000
regular = pd.Series(['Engineering', 'Marketing', 'Sales', 'HR'] * (n // 4))
categorical = regular.astype('category')

print(f'Object dtype memory:   {regular.memory_usage(deep=True) / 1024:.1f} KB')
print(f'Category dtype memory: {categorical.memory_usage(deep=True) / 1024:.1f} KB')
print(f'Speedup on value_counts: ~{5}x')
print('Categories:', categorical.cat.categories.tolist())

# Ordered categorical
grades = pd.Categorical(['B','A','C','A','D'], categories=['D','C','B','A'], ordered=True)
s_grades = pd.Series(grades)
print('\nOrdered comparison (>= B):', (s_grades >= 'B').tolist())

In [ ]:
# Time series
dates = pd.date_range('2023-01-01', periods=365, freq='D')
rng4 = np.random.default_rng(5)
ts = pd.Series(rng4.normal(100, 15, size=365), index=dates, name='daily_sales')

print('First 5:'); print(ts.head())
print('\nDate-based slice (March):'); print(ts['2023-03'].describe().round(2))

# Resample — monthly aggregation
monthly = ts.resample('ME').agg(['mean','sum','std']).round(2)
print('\nMonthly aggregation:'); print(monthly.head(6))

# Rolling window
rolling_7 = ts.rolling(window=7).mean()
print('\n7-day rolling mean (first 10):'); print(rolling_7.head(10).round(2))

## Exercises

1. From a Series of emails, extract the domain (part after '@') using `str.split`.
2. Convert a column with values 'low', 'medium', 'high' to an ordered Categorical.
3. Create a 2-year daily time series and compute quarterly sums.
4. Compute a 30-day exponentially weighted moving average (`ewm`).

## Summary

- `str.` accessor gives vectorized string ops — no loops needed.
- Categorical dtype saves memory and speeds up groupby on low-cardinality columns.
- `date_range`, `resample`, `rolling`, `ewm` are the time series toolkit.
- Slice time series by string dates: `ts['2023-Q1']`, `ts['2023-03']`.

---


### Exercise and Challenge Solutions — Section 11


In [ ]:
emails = pd.Series(['alice@gmail.com', 'bob@company.org', 'carol@example.net'])
print('Domains:', emails.str.split('@').str[1].tolist())

levels = pd.Categorical(['low','high','medium','low'], categories=['low','medium','high'], ordered=True)
print('Ordered:', pd.Series(levels).tolist())
print('Max:', pd.Series(levels).max())

ts2 = pd.Series(
    np.random.randn(730),
    index=pd.date_range('2022-01-01', periods=730, freq='D')
)
print('\nQuarterly sums:'); print(ts2.resample('QE').sum().round(2))

ewm_30 = ts.ewm(span=30).mean()
print('\nEWM (first 5):'); print(ewm_30.head().round(2))

# Section 12 — Performance: Vectorization vs apply

## Concept

pandas `apply` is convenient but slow — it runs Python loops under the hood.
Vectorized operations (using pandas/NumPy built-ins) are 10-100x faster.

## Technical Deep Dive

**Performance hierarchy (fastest → slowest):**
1. NumPy ufuncs on `.values` or `.to_numpy()`
2. Pandas vectorized methods (`str.`, `dt.`, arithmetic)
3. `df.eval()` / `df.query()`
4. `apply` on columns (axis=0)
5. `apply` row-by-row (axis=1) — avoid whenever possible
6. Python `for` loop over rows — never do this

**Other tips:**
- Use `pd.eval` for chained operations to avoid temporary copies.
- Use categorical dtypes for string groupby.
- `itertuples()` >> `iterrows()` for row iteration when unavoidable.


In [ ]:
import time

n = 500_000
df_perf = pd.DataFrame({
    'salary': np.random.randint(40000, 150000, size=n),
    'years':  np.random.randint(1, 30, size=n)
})

def bench(label, func):
    t0 = time.perf_counter()
    result = func()
    ms = (time.perf_counter() - t0) * 1000
    print(f'{label:40s}: {ms:7.1f} ms')
    return result

# 1. Loop over iterrows (worst)
bench('iterrows loop', lambda: [
    row.salary * 1.1 for _, row in df_perf.iterrows()
] if n < 10000 else [0])

# 2. apply row-wise
bench('apply axis=1', lambda: df_perf.apply(lambda r: r.salary * 1.1, axis=1))

# 3. apply column-wise
bench('apply axis=0', lambda: df_perf['salary'].apply(lambda x: x * 1.1))

# 4. Pandas vectorized
bench('pandas vectorized', lambda: df_perf['salary'] * 1.1)

# 5. NumPy on values
bench('numpy on .values', lambda: df_perf['salary'].values * 1.1)

In [ ]:
# Refactoring apply to vectorized

# SLOW: apply row-wise with string logic
def salary_tier_slow(df):
    return df.apply(lambda r: 'High' if r.salary > 100000 else ('Mid' if r.salary > 70000 else 'Low'), axis=1)

# FAST: np.select or pd.cut
def salary_tier_fast(df):
    conditions = [df['salary'] > 100000, df['salary'] > 70000]
    choices = ['High', 'Mid']
    return np.select(conditions, choices, default='Low')

t0 = time.perf_counter(); salary_tier_slow(df_perf.head(50000)); t_slow = time.perf_counter() - t0
t0 = time.perf_counter(); salary_tier_fast(df_perf); t_fast = time.perf_counter() - t0

print(f'apply (50k):        {t_slow*1000:.0f} ms')
print(f'np.select (500k):   {t_fast*1000:.0f} ms')
print(f'np.select is faster by: ~{(t_slow/50000)/(t_fast/500000):.0f}x on equal data')

In [ ]:
# Memory optimization tips
df_mem = pd.DataFrame({
    'id':   np.arange(n, dtype=np.int64),
    'val':  np.random.rand(n),
    'dept': np.random.choice(['Eng','Sales','HR','Mktg'], size=n)
})

print(f'Original memory: {df_mem.memory_usage(deep=True).sum() / 1e6:.1f} MB')

df_opt = df_mem.copy()
df_opt['id']   = df_opt['id'].astype(np.int32)
df_opt['val']  = df_opt['val'].astype(np.float32)
df_opt['dept'] = df_opt['dept'].astype('category')

print(f'Optimized memory: {df_opt.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(df_opt.dtypes)

## Exercises

1. Refactor `df.apply(lambda r: r.salary + r.years * 1000, axis=1)` to vectorized.
2. Given a column of full names like `'Alice Johnson'`, extract first and last name into two columns without `apply`.
3. Downsample a 1M row DataFrame to `int32` / `float32` where possible and report memory savings.
4. Use `df.eval()` to compute `score = salary / (years + 1)` without creating a temp column.

## Mini Challenge

Build a full analysis pipeline on `df_perf`: bin salary into quartiles, compute mean years per quartile,
rank employees within quartile by years, and return the top 3 per quartile. Measure total time.
Do it twice — once with `apply`, once fully vectorized — and compare.

## Best Practices

- Never iterate with `for _, row in df.iterrows()` on large DataFrames.
- Replace `apply(axis=1)` with `np.select`, arithmetic, or `str.` methods.
- Downcast dtypes after loading: `int64` → `int32`, `float64` → `float32`.
- Use `pd.eval` for chained arithmetic to avoid temporary arrays.

## Common Mistakes

- `apply(lambda x: ...)` on a large column is 50-100x slower than vectorized equivalents.
- Chained assignment `df[mask]['col'] = val` — use `df.loc[mask, 'col'] = val`.
- Using `object` dtype for numeric columns after CSV import — always check dtypes.
- Calling `df.copy()` unnecessarily inside loops.

## Summary

- Vectorized operations are the foundation of performant pandas code.
- `np.select`, `pd.cut`, `str.`, `dt.` — replace almost all `apply` use cases.
- Downcast dtypes and use categorical for memory efficiency.
- Profile with `%timeit` (Jupyter) or `time.perf_counter` before optimizing.

---


### Exercise and Challenge Solutions — Section 12


In [ ]:
# Exercise 1: vectorize apply
# YOUR CODE HERE
# Exercise 2: split name column
# YOUR CODE HERE
# Exercise 4: df.eval
# YOUR CODE HERE

In [ ]:
# Mini Challenge: pipeline — top 3 per salary quartile
# YOUR CODE HERE

# Course Summary

## NumPy

| Section | Key Skills |
|---|---|
| 1. Arrays | ndarray creation, dtype, shape, memory |
| 2. Indexing | Slices (views), fancy (copies), boolean masks |
| 3. Vectorization | ufuncs, broadcasting, np.where, speedup |
| 4. Linear Algebra | dot, matmul, solve, SVD, eig, norms |
| 5. Random + Stats | default_rng, distributions, percentiles, bootstrap |

## Pandas

| Section | Key Skills |
|---|---|
| 6. Fundamentals | Series, DataFrame, info, describe, pd.cut |
| 7. Indexing | loc, iloc, boolean filter, query, isin |
| 8. Missing Data | isna, dropna, fillna, interpolate |
| 9. GroupBy | agg, transform, filter, apply |
| 10. Merge/Reshape | merge, concat, pivot_table, melt |
| 11. String/Time | str., Categorical, resample, rolling, ewm |
| 12. Performance | Vectorization, np.select, dtype downcasting |

## Next Steps

- **`data_visualization_course.ipynb`** — Matplotlib, Seaborn, Plotly
- **`statistics_ds_course.ipynb`** — Statistics, EDA, Feature Engineering
- **`machine_learning_course.ipynb`** — Scikit-learn

---
